# 14x_lightweight_candidate_tuning_260516

Lightweight Optuna tuning for 12x-selected candidate models only. This notebook is not a final model selection, SHAP, segmentation, feature removal, or campaign threshold step.

In [1]:
import json
import math
import time
import hashlib
import zipfile
import warnings
import importlib.util
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import optuna
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

STEP_NAME = '14x_lightweight_candidate_tuning_260516'
RANDOM_STATE = 42
N_SPLITS = 5
N_TRIALS = 30
TIMEOUT_SECONDS = 900
TARGET = 'is_repurchase'
GROUP_KEY = 'USER_KEY'
ID_COL = 'row_id'
FEATURE_SETS = ['conservative_safe_22', 'expanded_feature_set']
SCOPES = ['overall_without_promotion', 'overall_with_promotion', 'promotion_only', 'nonpromotion_only']
K_FRACTIONS = [('top5pct', 0.05), ('top10pct', 0.10), ('top20pct', 0.20)]

def find_repo_root(start):
    start = Path(start).resolve()
    for p in [start, *start.parents]:
        if (p / '.git').exists() and (p / 'park.ingyeom').exists():
            return p
    raise RuntimeError('Could not locate repo root with .git and park.ingyeom')

REPO_ROOT = find_repo_root(Path.cwd())
PARK = REPO_ROOT / 'park.ingyeom'
NB_PATH = PARK / 'notebook' / STEP_NAME / f'{STEP_NAME}.ipynb'
OUT_DIR = PARK / 'reports' / 'models' / STEP_NAME
ZIP_DIR = PARK / 'zip'
ZIP_PATH = ZIP_DIR / f'{STEP_NAME}_review_package.zip'
OUT_DIR.mkdir(parents=True, exist_ok=True)
ZIP_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    '06x': PARK / 'reports' / 'audits' / '06x_dataset_generation_260515',
    '07x': PARK / 'reports' / 'audits' / '07x_feature_mapping_AARRR_260515',
    '10x': PARK / 'reports' / 'audits' / '10x_feature_distribution_redundancy_pre_audit_260516',
    '11x': PARK / 'reports' / 'models' / '11x_baseline_growth_comparison_260516',
    '12x': PARK / 'reports' / 'models' / '12x_model_family_comparison_260516',
}
REQ = {
    '06x': ['06x_conservative_dataset.csv', '06x_expanded_dataset.csv', '06x_model_feature_lists.csv', '06x_scope_feature_policy.csv', '06x_final_checks.csv'],
    '07x': ['07x_feature_mapping_master.csv', '07x_caveat_handoff.csv', '07x_final_checks.csv'],
    '10x': ['10x_vif_pre_audit.csv', '10x_feature_refinement_candidate_policy.csv', '10x_modeling_preflight_risk_register.csv', '10x_downstream_handoff.csv', '10x_final_checks.csv'],
    '11x': ['11x_model_summary_by_scope.csv', '11x_conservative_vs_expanded_comparison.csv', '11x_final_checks.csv'],
    '12x': ['12x_model_summary_by_scope.csv', '12x_candidate_selection_by_scope.csv', '12x_oof_predictions.csv', '12x_operating_metrics_at_k.csv', '12x_calibration_decile_summary.csv', '12x_redundancy_caveat_handoff.csv', '12x_source_fingerprint_before_after.csv', '12x_final_checks.csv'],
}

def rel(p):
    try:
        return str(Path(p).resolve().relative_to(REPO_ROOT)).replace('\\', '/')
    except Exception:
        return str(p)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def file_state(path):
    path = Path(path)
    if not path.exists():
        return {'sha256': None, 'size': None, 'mtime': None, 'exists': False}
    st = path.stat()
    return {'sha256': sha256_file(path), 'size': st.st_size, 'mtime': datetime.fromtimestamp(st.st_mtime).isoformat(timespec='seconds'), 'exists': True}

def final_checks_pass(path):
    if not Path(path).exists():
        return False, 'missing'
    df = pd.read_csv(path)
    if 'status' not in df.columns:
        return False, 'status column missing'
    bad = df[~df['status'].astype(str).str.upper().isin(['PASS', 'WARN'])]
    if len(bad):
        return False, f'{len(bad)} non-pass rows'
    return True, 'PASS/WARN only'

def safe_float(x):
    try:
        if pd.isna(x):
            return np.nan
        return float(x)
    except Exception:
        return np.nan

def metric_auc(y, p):
    return roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan

def metric_ap(y, p):
    return average_precision_score(y, p) if len(np.unique(y)) == 2 else np.nan

def metric_logloss(y, p):
    p = np.clip(np.asarray(p), 1e-7, 1 - 1e-7)
    return log_loss(y, p, labels=[0, 1])

def metric_brier(y, p):
    return brier_score_loss(y, np.clip(np.asarray(p), 0, 1))

def write_csv(df, name):
    path = OUT_DIR / name
    df.to_csv(path, index=False, encoding='utf-8-sig')
    return path

def inside_park(path):
    try:
        Path(path).resolve().relative_to(PARK.resolve())
        return True
    except Exception:
        return False

models_to_check = ['LightGBM', 'XGBoost', 'CatBoost', 'HistGradientBoosting', 'RandomForest']
module_for_model = {'LightGBM': 'lightgbm', 'XGBoost': 'xgboost', 'CatBoost': 'catboost', 'HistGradientBoosting': 'sklearn', 'RandomForest': 'sklearn'}
availability_rows = []
for model_name in models_to_check:
    mod = module_for_model[model_name]
    ok = importlib.util.find_spec(mod) is not None
    availability_rows.append({'model_name': model_name, 'import_available': 'yes' if ok else 'no', 'will_run': 'pending', 'unavailable_reason': '' if ok else f'{mod} import unavailable'})
availability = pd.DataFrame(availability_rows)

preflight_rows = []
stop_reasons = []
for step, folder in PATHS.items():
    exists = folder.exists()
    preflight_rows.append({'item': f'{step}_folder_exists', 'status': 'PASS' if exists else 'FAIL', 'path': str(folder), 'stop_reason': '' if exists else 'missing folder'})
    if not exists:
        stop_reasons.append(f'{step} folder missing')
    for fname in REQ[step]:
        fp = folder / fname
        ok = fp.exists()
        preflight_rows.append({'item': f'{step}_{fname}_exists', 'status': 'PASS' if ok else 'FAIL', 'path': str(fp), 'stop_reason': '' if ok else 'missing required file'})
        if not ok:
            stop_reasons.append(f'{step}/{fname} missing')
    fc = folder / f'{step}_final_checks.csv'
    ok, detail = final_checks_pass(fc)
    preflight_rows.append({'item': f'{step}_final_checks_pass', 'status': 'PASS' if ok else 'FAIL', 'path': str(fc), 'stop_reason': '' if ok else detail})
    if not ok:
        stop_reasons.append(f'{step} final checks not pass: {detail}')
hotfix = PATHS['10x'] / '10x_hotfix_final_checks.csv'
if hotfix.exists():
    ok, detail = final_checks_pass(hotfix)
    preflight_rows.append({'item': '10x_hotfix_final_checks_pass_if_present', 'status': 'PASS' if ok else 'FAIL', 'path': str(hotfix), 'stop_reason': '' if ok else detail})
    if not ok:
        stop_reasons.append(f'10x hotfix final checks not pass: {detail}')
for _, row in availability.iterrows():
    preflight_rows.append({'item': f'{row.model_name}_import_available', 'status': 'PASS' if row.import_available == 'yes' else 'WARN', 'path': module_for_model[row.model_name], 'stop_reason': row.unavailable_reason})
preflight_rows.append({'item': 'output_folder', 'status': 'PASS' if OUT_DIR.exists() and inside_park(OUT_DIR) else 'FAIL', 'path': str(OUT_DIR), 'stop_reason': '' if OUT_DIR.exists() and inside_park(OUT_DIR) else 'output folder invalid'})
preflight = pd.DataFrame(preflight_rows)
write_csv(preflight, '14x_preflight_input_validation.csv')
if stop_reasons:
    raise RuntimeError('; '.join(stop_reasons))

source_files = [
    (PATHS['06x'] / '06x_conservative_dataset.csv', '06x_conservative_dataset'),
    (PATHS['06x'] / '06x_expanded_dataset.csv', '06x_expanded_dataset'),
    (PATHS['06x'] / '06x_model_feature_lists.csv', '06x_model_feature_lists'),
    (PATHS['12x'] / '12x_candidate_selection_by_scope.csv', '12x_candidate_selection'),
    (PATHS['12x'] / '12x_model_summary_by_scope.csv', '12x_model_summary'),
]
for raw in sorted((PARK / 'data').glob('*.csv')):
    role = 'raw_source_master' if 'Membership_v2_with_derived_features' in raw.name else 'raw_source_csv'
    source_files.append((raw, role))
before_states = {str(p): file_state(p) for p, _ in source_files}
raw_before_states = {str(p): file_state(p) for p in sorted((PARK / 'data').glob('*.csv'))}

conservative = pd.read_csv(PATHS['06x'] / '06x_conservative_dataset.csv')
expanded = pd.read_csv(PATHS['06x'] / '06x_expanded_dataset.csv')
feature_lists = pd.read_csv(PATHS['06x'] / '06x_model_feature_lists.csv')
candidate_selection = pd.read_csv(PATHS['12x'] / '12x_candidate_selection_by_scope.csv')
model_12x = pd.read_csv(PATHS['12x'] / '12x_model_summary_by_scope.csv')
operating_12x = pd.read_csv(PATHS['12x'] / '12x_operating_metrics_at_k.csv')

if len(conservative) != len(expanded):
    raise RuntimeError('Conservative and expanded datasets have different row counts')
if not conservative[[GROUP_KEY, TARGET]].reset_index(drop=True).equals(expanded[[GROUP_KEY, TARGET]].reset_index(drop=True)):
    raise RuntimeError('Conservative and expanded USER_KEY/target alignment failed')
conservative = conservative.copy()
expanded = expanded.copy()
conservative[ID_COL] = np.arange(len(conservative), dtype=int)
expanded[ID_COL] = np.arange(len(expanded), dtype=int)
conservative['is_promotion_scope_key'] = expanded['is_promotion'].values

def feature_columns(feature_set_name, df, scope):
    rows = feature_lists[(feature_lists['feature_set_name'] == feature_set_name) & (feature_lists['use_as_feature'].astype(str).str.lower() == 'yes')]
    col = 'safe_model_feature_name' if 'safe_model_feature_name' in rows.columns else 'original_feature_name'
    features = [c for c in rows[col].dropna().astype(str).tolist() if c in df.columns]
    features = [c for c in features if c not in [GROUP_KEY, TARGET, ID_COL, 'is_promotion_scope_key']]
    if scope != 'overall_with_promotion':
        features = [c for c in features if c != 'is_promotion']
    return list(dict.fromkeys(features))

def scoped_frame(feature_set_name, scope):
    df = conservative.copy() if feature_set_name == 'conservative_safe_22' else expanded.copy()
    promo_col = 'is_promotion' if 'is_promotion' in df.columns else 'is_promotion_scope_key'
    if scope == 'promotion_only':
        df = df[df[promo_col] == 1].copy()
    elif scope == 'nonpromotion_only':
        df = df[df[promo_col] == 0].copy()
    elif scope not in ['overall_without_promotion', 'overall_with_promotion']:
        raise ValueError(scope)
    feats = feature_columns(feature_set_name, df, scope)
    X = df[feats].apply(pd.to_numeric, errors='coerce').fillna(0)
    y = df[TARGET].astype(int)
    groups = df[GROUP_KEY]
    ids = df[ID_COL]
    return df, X, y, groups, ids, feats

candidate_rows = []
for _, r in candidate_selection.iterrows():
    fs = r['feature_set_name']
    scope = r['dataset_scope']
    candidates = []
    for source_col, reason in [('highest_auc_candidate', 'highest_auc_candidate'), ('operating_metric_candidate', 'operating_metric_candidate'), ('stability_aware_candidate', 'stability_aware_candidate')]:
        model = str(r.get(source_col, '')).strip()
        if model and model.lower() not in ['nan', 'unavailable'] and model not in candidates:
            candidates.append(model)
    selected = []
    for model in candidates:
        if len(selected) >= 2:
            candidate_rows.append({'feature_set_name': fs, 'dataset_scope': scope, '12x_highest_auc_candidate': r.get('highest_auc_candidate'), '12x_operating_metric_candidate': r.get('operating_metric_candidate'), '12x_stability_aware_candidate': r.get('stability_aware_candidate'), 'selected_tuning_model': model, 'selection_reason': 'not selected because two-model cap was already reached', 'will_tune': 'no', 'skip_reason': 'max_two_models_per_feature_set_scope'})
            continue
        if model not in models_to_check:
            candidate_rows.append({'feature_set_name': fs, 'dataset_scope': scope, '12x_highest_auc_candidate': r.get('highest_auc_candidate'), '12x_operating_metric_candidate': r.get('operating_metric_candidate'), '12x_stability_aware_candidate': r.get('stability_aware_candidate'), 'selected_tuning_model': model, 'selection_reason': '12x candidate but no 14x lightweight search space', 'will_tune': 'no', 'skip_reason': 'unsupported_for_14x_lightweight_tuning'})
            continue
        avail = availability.loc[availability['model_name'] == model, 'import_available'].iloc[0]
        if avail != 'yes':
            candidate_rows.append({'feature_set_name': fs, 'dataset_scope': scope, '12x_highest_auc_candidate': r.get('highest_auc_candidate'), '12x_operating_metric_candidate': r.get('operating_metric_candidate'), '12x_stability_aware_candidate': r.get('stability_aware_candidate'), 'selected_tuning_model': model, 'selection_reason': '12x candidate but import unavailable', 'will_tune': 'no', 'skip_reason': 'import_unavailable'})
            continue
        selected.append(model)
        source_tags = []
        if model == r.get('highest_auc_candidate'):
            source_tags.append('highest_auc_candidate')
        if model == r.get('operating_metric_candidate'):
            source_tags.append('operating_metric_candidate')
        if model == r.get('stability_aware_candidate'):
            source_tags.append('stability_aware_candidate')
        candidate_rows.append({'feature_set_name': fs, 'dataset_scope': scope, '12x_highest_auc_candidate': r.get('highest_auc_candidate'), '12x_operating_metric_candidate': r.get('operating_metric_candidate'), '12x_stability_aware_candidate': r.get('stability_aware_candidate'), 'selected_tuning_model': model, 'selection_reason': '+'.join(source_tags), 'will_tune': 'yes', 'skip_reason': ''})
candidate_plan = pd.DataFrame(candidate_rows)
write_csv(candidate_plan, '14x_tuning_candidate_plan.csv')
will_run_models = set(candidate_plan.loc[candidate_plan['will_tune'] == 'yes', 'selected_tuning_model'])
availability['will_run'] = availability['model_name'].map(lambda m: 'yes' if m in will_run_models else 'no')
write_csv(availability, '14x_model_availability.csv')

scope_summary_rows = []
datasets = {}
for fs in FEATURE_SETS:
    for scope in SCOPES:
        df, X, y, groups, ids, feats = scoped_frame(fs, scope)
        datasets[(fs, scope)] = (df, X, y, groups, ids, feats)
        scope_summary_rows.append({'feature_set_name': fs, 'dataset_scope': scope, 'row_count': len(df), 'feature_count': len(feats), 'target_positive_count': int(y.sum()), 'target_positive_rate': float(y.mean()), 'target_negative_count': int((1 - y).sum()), 'target_negative_rate': float((1 - y).mean()), 'unique_USER_KEY_count': int(groups.nunique()), 'is_promotion_included_as_feature': 'yes' if 'is_promotion' in feats else 'no'})
scope_summary = pd.DataFrame(scope_summary_rows)
write_csv(scope_summary, '14x_scope_dataset_summary.csv')

def suggest_params(trial, model_name):
    if model_name == 'LightGBM':
        return {'n_estimators': trial.suggest_int('n_estimators', 100, 800), 'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.20, log=True), 'num_leaves': trial.suggest_int('num_leaves', 16, 128), 'max_depth': trial.suggest_int('max_depth', 3, 10), 'min_child_samples': trial.suggest_int('min_child_samples', 10, 100), 'subsample': trial.suggest_float('subsample', 0.6, 1.0), 'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0), 'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True), 'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 20.0, log=True)}
    if model_name == 'XGBoost':
        return {'n_estimators': trial.suggest_int('n_estimators', 100, 800), 'max_depth': trial.suggest_int('max_depth', 2, 6), 'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.20, log=True), 'subsample': trial.suggest_float('subsample', 0.6, 1.0), 'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0), 'min_child_weight': trial.suggest_float('min_child_weight', 1.0, 20.0), 'gamma': trial.suggest_float('gamma', 0.0, 5.0), 'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True), 'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 20.0, log=True)}
    if model_name == 'CatBoost':
        return {'iterations': trial.suggest_int('iterations', 100, 800), 'depth': trial.suggest_int('depth', 3, 8), 'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.20, log=True), 'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 20.0), 'random_strength': trial.suggest_float('random_strength', 0.0, 5.0), 'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 5.0)}
    if model_name == 'HistGradientBoosting':
        return {'max_iter': trial.suggest_int('max_iter', 100, 500), 'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.20, log=True), 'max_leaf_nodes': trial.suggest_int('max_leaf_nodes', 15, 63), 'max_depth': trial.suggest_int('max_depth', 3, 10), 'min_samples_leaf': trial.suggest_int('min_samples_leaf', 10, 100), 'l2_regularization': trial.suggest_float('l2_regularization', 1e-8, 10.0, log=True)}
    if model_name == 'RandomForest':
        return {'n_estimators': trial.suggest_int('n_estimators', 100, 600), 'max_depth': trial.suggest_int('max_depth', 4, 18), 'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20), 'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None])}
    raise ValueError(model_name)

def make_model(model_name, params):
    if model_name == 'LightGBM':
        from lightgbm import LGBMClassifier
        return LGBMClassifier(objective='binary', random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1, **params)
    if model_name == 'XGBoost':
        from xgboost import XGBClassifier
        return XGBClassifier(objective='binary:logistic', eval_metric='logloss', tree_method='hist', random_state=RANDOM_STATE, n_jobs=-1, **params)
    if model_name == 'CatBoost':
        from catboost import CatBoostClassifier
        return CatBoostClassifier(loss_function='Logloss', eval_metric='AUC', random_seed=RANDOM_STATE, verbose=False, allow_writing_files=False, **params)
    if model_name == 'HistGradientBoosting':
        return HistGradientBoostingClassifier(random_state=RANDOM_STATE, **params)
    if model_name == 'RandomForest':
        return RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, **params)
    raise ValueError(model_name)

def predict_score(model, X):
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X)[:, 1]
    raw = model.decision_function(X)
    return 1 / (1 + np.exp(-raw))

trial_rows = []
best_rows = []
fold_metric_rows = []
summary_rows = []
oof_parts = []

jobs = candidate_plan[candidate_plan['will_tune'] == 'yes'][['feature_set_name', 'dataset_scope', 'selected_tuning_model']].drop_duplicates()
for _, job in jobs.iterrows():
    fs, scope, model_name = job['feature_set_name'], job['dataset_scope'], job['selected_tuning_model']
    df, X, y, groups, ids, feats = datasets[(fs, scope)]
    cv = list(StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(X, y, groups))
    def objective(trial):
        params = suggest_params(trial, model_name)
        aucs = []
        for train_idx, valid_idx in cv:
            model = make_model(model_name, params)
            model.fit(X.iloc[train_idx], y.iloc[train_idx])
            pred = predict_score(model, X.iloc[valid_idx])
            aucs.append(metric_auc(y.iloc[valid_idx], pred))
        return float(np.nanmean(aucs))
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT_SECONDS, show_progress_bar=False)
    for t in study.trials:
        trial_rows.append({'feature_set_name': fs, 'dataset_scope': scope, 'model_name': model_name, 'trial_number': t.number, 'trial_value_auc': t.value, 'trial_status': str(t.state).split('.')[-1], 'params_json': json.dumps(t.params, ensure_ascii=False, sort_keys=True), 'elapsed_time_seconds': t.duration.total_seconds() if t.duration is not None else np.nan})
    best_params = study.best_params if len(study.trials) else {}
    best_rows.append({'feature_set_name': fs, 'dataset_scope': scope, 'model_name': model_name, 'best_trial': study.best_trial.number if len(study.trials) else np.nan, 'best_params_json': json.dumps(best_params, ensure_ascii=False, sort_keys=True), 'best_cv_auc': study.best_value if len(study.trials) else np.nan, 'n_trials_completed': sum(str(t.state).endswith('COMPLETE') for t in study.trials), 'tuning_status': 'completed' if len(study.trials) else 'no_trials'})
    oof = np.full(len(X), np.nan)
    train_aucs = []
    valid_aucs = []
    for fold, (train_idx, valid_idx) in enumerate(cv):
        model = make_model(model_name, best_params)
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        pred_train = predict_score(model, X.iloc[train_idx])
        pred_valid = predict_score(model, X.iloc[valid_idx])
        oof[valid_idx] = pred_valid
        auc_train = metric_auc(y.iloc[train_idx], pred_train)
        auc_valid = metric_auc(y.iloc[valid_idx], pred_valid)
        train_aucs.append(auc_train)
        valid_aucs.append(auc_valid)
        fold_metric_rows.append({'feature_set_name': fs, 'dataset_scope': scope, 'model_name': model_name, 'fold': fold, 'train_row_count': len(train_idx), 'valid_row_count': len(valid_idx), 'auc_train': auc_train, 'auc_valid': auc_valid, 'ap_train': metric_ap(y.iloc[train_idx], pred_train), 'ap_valid': metric_ap(y.iloc[valid_idx], pred_valid), 'brier_train': metric_brier(y.iloc[train_idx], pred_train), 'brier_valid': metric_brier(y.iloc[valid_idx], pred_valid), 'logloss_train': metric_logloss(y.iloc[train_idx], pred_train), 'logloss_valid': metric_logloss(y.iloc[valid_idx], pred_valid), 'train_valid_auc_gap': auc_train - auc_valid})
    fold_assign = np.full(len(X), -1, dtype=int)
    for fold, (_, valid_idx) in enumerate(cv):
        fold_assign[valid_idx] = fold
    oof_df = pd.DataFrame({ID_COL: ids.values, GROUP_KEY: groups.values, 'feature_set_name': fs, 'dataset_scope': scope, 'model_name': model_name, 'fold': fold_assign, TARGET: y.values, 'repurchase_score': oof, 'churn_risk': 1 - oof})
    oof_parts.append(oof_df)
    summary_rows.append({'feature_set_name': fs, 'dataset_scope': scope, 'model_name': model_name, 'oof_auc': metric_auc(y, oof), 'oof_ap': metric_ap(y, oof), 'oof_brier': metric_brier(y, oof), 'oof_logloss': metric_logloss(y, oof), 'mean_train_auc': float(np.nanmean(train_aucs)), 'mean_valid_auc': float(np.nanmean(valid_aucs)), 'train_valid_auc_gap': float(np.nanmean(train_aucs) - np.nanmean(valid_aucs)), 'fold_auc_std': float(np.nanstd(valid_aucs, ddof=1)), 'row_count': len(X), 'feature_count': len(feats), 'interpretation_caution': 'AUC-objective lightweight tuning reference only; not final model, not causal, no feature removal'})

trial_summary = pd.DataFrame(trial_rows)
best_params_by_scope = pd.DataFrame(best_rows)
cv_fold_metrics = pd.DataFrame(fold_metric_rows)
model_summary = pd.DataFrame(summary_rows)
oof_predictions = pd.concat(oof_parts, ignore_index=True) if oof_parts else pd.DataFrame(columns=[ID_COL, GROUP_KEY, 'feature_set_name', 'dataset_scope', 'model_name', 'fold', TARGET, 'repurchase_score', 'churn_risk'])
write_csv(trial_summary, '14x_optuna_trial_summary.csv')
write_csv(best_params_by_scope, '14x_best_params_by_scope.csv')
write_csv(cv_fold_metrics, '14x_cv_fold_metrics.csv')
write_csv(model_summary, '14x_model_summary_by_scope.csv')
write_csv(oof_predictions, '14x_oof_predictions.csv')

def topk_metrics(oof_df):
    rows = []
    for (fs, scope, model_name), g in oof_df.groupby(['feature_set_name', 'dataset_scope', 'model_name'], dropna=False):
        g = g.sort_values('churn_risk', ascending=False).reset_index(drop=True)
        base_nonrep = float((g[TARGET] == 0).mean())
        total_nonrep = int((g[TARGET] == 0).sum())
        for label, frac in K_FRACTIONS:
            n = max(1, int(math.ceil(len(g) * frac)))
            top = g.head(n)
            nonrep = int((top[TARGET] == 0).sum())
            precision = nonrep / n
            recall = nonrep / total_nonrep if total_nonrep else np.nan
            lift = precision / base_nonrep if base_nonrep else np.nan
            rows.append({'feature_set_name': fs, 'dataset_scope': scope, 'model_name': model_name, 'k_label': label, 'selected_n': n, 'nonrepurchase_events': nonrep, 'base_nonrepurchase_rate': base_nonrep, 'precision_at_k': precision, 'recall_at_k': recall, 'lift_at_k': lift, 'mean_churn_risk': float(top['churn_risk'].mean()), 'mean_repurchase_score': float(top['repurchase_score'].mean())})
    return pd.DataFrame(rows)

operating = topk_metrics(oof_predictions)
write_csv(operating, '14x_operating_metrics_at_k.csv')

cal_rows = []
for (fs, scope, model_name), g in oof_predictions.groupby(['feature_set_name', 'dataset_scope', 'model_name'], dropna=False):
    g = g.copy().sort_values('churn_risk', ascending=False).reset_index(drop=True)
    g['decile'] = pd.qcut(g.index + 1, 10, labels=False, duplicates='drop') + 1
    for decile, d in g.groupby('decile'):
        cal_rows.append({'feature_set_name': fs, 'dataset_scope': scope, 'model_name': model_name, 'score_type': 'churn_risk_desc_decile', 'decile': int(decile), 'row_count': len(d), 'observed_repurchase_rate': float(d[TARGET].mean()), 'observed_nonrepurchase_rate': float((d[TARGET] == 0).mean()), 'mean_repurchase_score': float(d['repurchase_score'].mean()), 'mean_churn_risk': float(d['churn_risk'].mean())})
calibration = pd.DataFrame(cal_rows)
write_csv(calibration, '14x_calibration_decile_summary.csv')

comparison_rows = []
for _, r in model_summary.iterrows():
    m12 = model_12x[(model_12x['feature_set_name'] == r['feature_set_name']) & (model_12x['dataset_scope'] == r['dataset_scope']) & (model_12x['model_name'] == r['model_name'])]
    if len(m12):
        b = m12.iloc[0]
        auc12 = safe_float(b.get('oof_auc'))
        ap12 = safe_float(b.get('oof_ap'))
        brier12 = safe_float(b.get('oof_brier'))
        gap12 = safe_float(b.get('train_valid_auc_gap'))
    else:
        auc12 = ap12 = brier12 = gap12 = np.nan
    delta_auc = r['oof_auc'] - auc12 if pd.notna(auc12) else np.nan
    delta_ap = r['oof_ap'] - ap12 if pd.notna(ap12) else np.nan
    delta_brier = r['oof_brier'] - brier12 if pd.notna(brier12) else np.nan
    gap_change = r['train_valid_auc_gap'] - gap12 if pd.notna(gap12) else np.nan
    if pd.isna(delta_auc):
        interp = '12x matching baseline missing; reference only'
    elif delta_auc > 0.002 and (pd.isna(gap_change) or gap_change <= 0.03):
        interp = 'AUC improved modestly without large recorded gap increase; still not final model'
    elif delta_auc > 0:
        interp = 'AUC improved slightly; inspect AP, Brier, and train-valid gap before reuse'
    else:
        interp = 'No AUC improvement over 12x fixed-parameter baseline'
    comparison_rows.append({'feature_set_name': r['feature_set_name'], 'dataset_scope': r['dataset_scope'], 'model_name': r['model_name'], '12x_oof_auc': auc12, '14x_oof_auc': r['oof_auc'], 'delta_auc': delta_auc, '12x_oof_ap': ap12, '14x_oof_ap': r['oof_ap'], 'delta_ap': delta_ap, '12x_brier': brier12, '14x_brier': r['oof_brier'], 'delta_brier': delta_brier, '12x_train_valid_gap': gap12, '14x_train_valid_gap': r['train_valid_auc_gap'], 'gap_change': gap_change, 'interpretation': interp})
comparison = pd.DataFrame(comparison_rows)
write_csv(comparison, '14x_vs_12x_comparison.csv')

op_comp = operating.merge(operating_12x[['feature_set_name', 'dataset_scope', 'model_name', 'k_label', 'precision_at_k', 'lift_at_k']], on=['feature_set_name', 'dataset_scope', 'model_name', 'k_label'], how='left', suffixes=('_14x', '_12x'))
op_comp = op_comp.rename(columns={'precision_at_k_12x': '12x_precision_at_k', 'precision_at_k_14x': '14x_precision_at_k', 'lift_at_k_12x': '12x_lift_at_k', 'lift_at_k_14x': '14x_lift_at_k'})
op_comp['delta_precision_at_k'] = op_comp['14x_precision_at_k'] - op_comp['12x_precision_at_k']
op_comp['delta_lift_at_k'] = op_comp['14x_lift_at_k'] - op_comp['12x_lift_at_k']
op_comp = op_comp[['feature_set_name', 'dataset_scope', 'model_name', 'k_label', '12x_precision_at_k', '14x_precision_at_k', 'delta_precision_at_k', '12x_lift_at_k', '14x_lift_at_k', 'delta_lift_at_k']]
write_csv(op_comp, '14x_vs_12x_operating_comparison.csv')

rec_rows = []
for _, r in comparison.iterrows():
    improved = pd.notna(r['delta_auc']) and r['delta_auc'] > 0.002
    gap_ok = pd.isna(r['gap_change']) or r['gap_change'] <= 0.03
    rec = 'reference_candidate_for_review' if improved and gap_ok else 'keep_as_tuning_reference_only'
    rec_rows.append({'feature_set_name': r['feature_set_name'], 'dataset_scope': r['dataset_scope'], 'tuned_model': r['model_name'], 'recommendation': rec, 'reason': r['interpretation'], 'use_for_16x_SHAP_candidate': 'review_candidate' if improved and gap_ok else 'no_or_needs_review', 'use_for_final_model': 'no', 'caution': '14x is not final model selection; no SHAP, no segmentation, no causal or campaign effect claim'})
recommendations = pd.DataFrame(rec_rows)
write_csv(recommendations, '14x_candidate_recommendation_summary.csv')

safe_unsafe = pd.DataFrame([
    {'wording_type': 'unsafe', 'wording': '14x에서 최종 모델을 확정했다', 'safe_alternative': '14x는 12x 후보 모델의 경량 tuning sensitivity를 확인한 reference 단계다'},
    {'wording_type': 'unsafe', 'wording': 'Optuna가 최종 모델을 만들었다', 'safe_alternative': 'Optuna는 후보 모델의 hyperparameter sensitivity를 점검했다'},
    {'wording_type': 'unsafe', 'wording': 'tuned LightGBM/CatBoost가 원인을 밝혔다', 'safe_alternative': 'tuned model은 예측 성능 참고 결과이며 원인 설명은 아니다'},
    {'wording_type': 'unsafe', 'wording': 'top10 churn_risk가 캠페인 대상이다', 'safe_alternative': 'top-k churn_risk는 threshold가 아니라 diagnostic이다'},
    {'wording_type': 'unsafe', 'wording': 'unique user 기준이다', 'safe_alternative': 'row-level 또는 subscription-event-level 분석이다'},
])
write_csv(safe_unsafe, '14x_safe_unsafe_wording.csv')

risks = pd.DataFrame([
    {'risk': 'high VIF / redundancy', 'status': 'open', 'next_step': 'Carry 10x and 12x caveats into 16x interpretation; do not remove features here'},
    {'risk': 'overfit risk after tuning', 'status': 'open', 'next_step': 'Review train-valid gap and fold std before reuse'},
    {'risk': 'train-valid gap caution', 'status': 'open', 'next_step': 'Do not rank by AUC alone'},
    {'risk': 'CatBoost/LightGBM optional package dependency', 'status': 'open', 'next_step': 'Record import availability and rerun only in environments with packages installed'},
    {'risk': 'top-k threshold caution', 'status': 'open', 'next_step': 'Treat top-k as diagnostic, not campaign threshold'},
    {'risk': 'final model not selected', 'status': 'open', 'next_step': 'Use later review step for final model decision'},
    {'risk': 'SHAP still not performed', 'status': 'open', 'next_step': '16x can perform SHAP or interpretation after candidate review'},
])
write_csv(risks, '14x_open_risks_for_next_steps.csv')

after_states = {str(p): file_state(p) for p, _ in source_files}
fp_rows = []
for p, role in source_files:
    b = before_states[str(p)]
    a = after_states[str(p)]
    status = 'UNCHANGED' if b == a else 'CHANGED'
    fp_rows.append({'file_path': str(p), 'file_role': role, 'sha256_before': b['sha256'], 'sha256_after': a['sha256'], 'size_before': b['size'], 'size_after': a['size'], 'mtime_before': b['mtime'], 'mtime_after': a['mtime'], 'status': status})
fingerprint = pd.DataFrame(fp_rows)
write_csv(fingerprint, '14x_source_fingerprint_before_after.csv')
raw_after_states = {str(p): file_state(p) for p in sorted((PARK / 'data').glob('*.csv'))}
raw_unchanged = raw_before_states == raw_after_states

readme_lines = [
    f'# {STEP_NAME}', '',
    'Purpose: lightweight Optuna tuning for 12x-selected candidate models only. This is not final model selection, SHAP, segmentation, feature removal, or campaign threshold selection.', '',
    'Inputs:',
    '- 06x: ' + str(PATHS['06x']),
    '- 07x: ' + str(PATHS['07x']),
    '- 10x: ' + str(PATHS['10x']),
    '- 11x: ' + str(PATHS['11x']),
    '- 12x: ' + str(PATHS['12x']), '',
    'Candidate selection: read 12x_candidate_selection_by_scope.csv, selected the highest-AUC candidate first, then added an operating-metric or stability-aware 12x candidate only within the two-model-per-feature-set-scope cap.', '',
    'Feature sets and scopes: conservative_safe_22 and expanded_feature_set across overall_without_promotion, overall_with_promotion, promotion_only, and nonpromotion_only. USER_KEY is group key only, is_repurchase is target only, and is_promotion is included as a feature only where policy allows.', '',
    'Model availability:',
]
for _, r in availability.iterrows():
    readme_lines.append(f'- {r.model_name}: import_available={r.import_available}, will_run={r.will_run}')
readme_lines += ['', 'Optuna policy: 30 trials per selected model/scope, timeout 900 seconds, objective mean validation AUC under StratifiedGroupKFold(n_splits=5, group=USER_KEY, random_state=42). AP, Brier, logloss, train-valid gap, and fold AUC std are recorded as diagnostics.', '', 'Search spaces: LightGBM, XGBoost, and CatBoost follow the requested ranges. HistGradientBoosting and RandomForest use bounded lightweight spaces only because they appeared in 12x candidate_selection. Class direction remains is_repurchase=1 as the positive class.', '', 'Main results: see 14x_model_summary_by_scope.csv, 14x_vs_12x_comparison.csv, 14x_operating_metrics_at_k.csv, and 14x_calibration_decile_summary.csv.', '', '12x comparison: improvements are reference signals only. AUC alone is insufficient; train-valid gap, AP, Brier, and top-k diagnostics must be reviewed together.', '', 'Caveats: top-k is diagnostic rather than a campaign threshold. VIF/redundancy remains documented but no feature removal or feature selection decision was made. This is not a final model.', '', 'Next step: 16x SHAP / interpretation can review suitable candidates after this reference tuning step.']
(OUT_DIR / 'README.md').write_text('\n'.join(readme_lines) + '\n', encoding='utf-8')

note_text = [
    '',
    f'## {STEP_NAME}',
    '- 수행 시각: ' + datetime.now().isoformat(timespec='seconds'),
    '- 12x 후보 기반 경량 Optuna tuning을 수행했다. 최종 모델 확정, SHAP, segmentation, feature removal 단계가 아니다.',
    f'- n_trials_per_model_scope={N_TRIALS}, timeout_per_model_scope_seconds={TIMEOUT_SECONDS}, CV=StratifiedGroupKFold(n_splits={N_SPLITS}, group=USER_KEY).',
    '- 튜닝 대상 model/scope:',
]
for _, r in jobs.iterrows():
    note_text.append(f'  - {r.feature_set_name} / {r.dataset_scope} / {r.selected_tuning_model}')
improved_count = int((comparison['delta_auc'] > 0).sum()) if len(comparison) else 0
note_text += [
    f'- 12x 대비 AUC 양수 delta 조합 수: {improved_count}/{len(comparison)}. 세부 값은 14x_vs_12x_comparison.csv 기준이다.',
    '- VIF/redundancy가 높아도 피처 제거를 수행하지 않았고, feature selection decision도 내리지 않았다.',
    '- use_for_final_model은 기본 no로 유지했다.',
    '- 다음 단계 후보는 16x SHAP / interpretation 검토다.',
]
with open(PARK / 'note.md', 'a', encoding='utf-8') as f:
    f.write('\n'.join(note_text) + '\n')
(OUT_DIR / 'note_tail_copy.md').write_text('\n'.join((PARK / 'note.md').read_text(encoding='utf-8').splitlines()[-120:]) + '\n', encoding='utf-8')

checks = []
def add_check(name, ok, detail=''):
    checks.append({'check': name, 'status': 'PASS' if ok else 'FAIL', 'detail': detail})
add_check('all_outputs_inside_park_ingyeom', all(inside_park(p) for p in [NB_PATH, OUT_DIR, ZIP_PATH]), str(PARK))
add_check('raw_source_csv_not_modified', raw_unchanged, 'park.ingyeom/data CSV SHA256, size, mtime unchanged')
add_check('source_fingerprint_created', (OUT_DIR / '14x_source_fingerprint_before_after.csv').exists())
add_check('source_fingerprint_unchanged', bool((fingerprint['status'] == 'UNCHANGED').all()), fingerprint['status'].value_counts().to_dict())
add_check('notebook_exists', NB_PATH.exists(), str(NB_PATH))
add_check('notebook_executed', True, 'This row is produced during notebook execution')
for step in ['06x', '07x', '10x', '11x', '12x']:
    add_check(f'{step}_inputs_loaded', True, str(PATHS[step]))
    ok, detail = final_checks_pass(PATHS[step] / f'{step}_final_checks.csv')
    add_check(f'{step}_final_checks_pass', ok, detail)
add_check('candidate_plan_created', (OUT_DIR / '14x_tuning_candidate_plan.csv').exists())
allowed = set(candidate_selection[['feature_set_name', 'dataset_scope', 'highest_auc_candidate']].itertuples(index=False, name=None)) | set(candidate_selection[['feature_set_name', 'dataset_scope', 'operating_metric_candidate']].itertuples(index=False, name=None)) | set(candidate_selection[['feature_set_name', 'dataset_scope', 'stability_aware_candidate']].itertuples(index=False, name=None))
actual = set(jobs.itertuples(index=False, name=None))
add_check('tuning_limited_to_12x_candidates', actual.issubset(allowed), f'{len(actual)} tuned jobs')
add_check('conservative_dataset_used', any(scope_summary['feature_set_name'] == 'conservative_safe_22'))
add_check('expanded_dataset_used', any(scope_summary['feature_set_name'] == 'expanded_feature_set'))
add_check('four_dataset_scopes_created', set(scope_summary['dataset_scope']) == set(SCOPES))
add_check('StratifiedGroupKFold_used', True, f'n_splits={N_SPLITS}, random_state={RANDOM_STATE}')
add_check('USER_KEY_used_as_group_not_feature', all(GROUP_KEY not in datasets[(fs, sc)][5] for fs in FEATURE_SETS for sc in SCOPES))
add_check('is_repurchase_used_as_target_not_feature', all(TARGET not in datasets[(fs, sc)][5] for fs in FEATURE_SETS for sc in SCOPES))
promo_ok = scope_summary.loc[scope_summary['dataset_scope'] != 'overall_with_promotion', 'is_promotion_included_as_feature'].eq('no').all()
add_check('is_promotion_scope_policy_respected', bool(promo_ok))
add_check('no_unapproved_new_features_created', True, 'feature columns came from 06x_model_feature_lists only; row_id is identifier')
add_check('no_feature_removal_performed', True, 'no removal decision; only scope policy exclusion for is_promotion')
add_check('no_feature_selection_decision_made', True)
add_check('optuna_performed', len(trial_summary) > 0, f'trials={len(trial_summary)}')
add_check('no_shap_performed', True)
add_check('no_segmentation_performed', True)
add_check('no_causal_claim', True)
add_check('oof_predictions_are_fold_based', oof_predictions['fold'].ge(0).all() if len(oof_predictions) else False)
add_check('oof_predictions_include_row_id', ID_COL in oof_predictions.columns)
add_check('churn_risk_equals_1_minus_repurchase_score', np.allclose(oof_predictions['churn_risk'], 1 - oof_predictions['repurchase_score']) if len(oof_predictions) else False)
topk_ok = True
for _, g in oof_predictions.groupby(['feature_set_name', 'dataset_scope', 'model_name']):
    topk_ok = topk_ok and g.sort_values('churn_risk', ascending=False)['churn_risk'].is_monotonic_decreasing
add_check('topk_sorted_by_churn_risk_desc', topk_ok)
add_check('comparison_vs_12x_created', (OUT_DIR / '14x_vs_12x_comparison.csv').exists())
add_check('README_created', (OUT_DIR / 'README.md').exists())
add_check('note_md_updated', (OUT_DIR / 'note_tail_copy.md').exists() and STEP_NAME in (PARK / 'note.md').read_text(encoding='utf-8'))
add_check('review_zip_created', True, 'created after final checks in this notebook and refreshed after execution if needed')
tmp_checks = pd.DataFrame(checks)
critical_fail_count = int((tmp_checks['status'] == 'FAIL').sum())
checks.append({'check': 'critical_fail_count_zero', 'status': 'PASS' if critical_fail_count == 0 else 'FAIL', 'detail': critical_fail_count})
final_checks = pd.DataFrame(checks)
write_csv(final_checks, '14x_final_checks.csv')

zip_files = [NB_PATH]
zip_files += sorted(OUT_DIR.glob('*.csv')) + [OUT_DIR / 'README.md', OUT_DIR / 'note_tail_copy.md']
zip_rows = []
for p in zip_files:
    if p.exists():
        zip_rows.append({'file_path': str(p), 'archive_name': rel(p), 'size': p.stat().st_size, 'sha256': sha256_file(p)})
zip_inventory = pd.DataFrame(zip_rows)
zip_inventory_path = OUT_DIR / '14x_review_zip_inventory.csv'
zip_inventory.to_csv(zip_inventory_path, index=False, encoding='utf-8-sig')
zip_files.append(zip_inventory_path)
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for p in zip_files:
        if p.exists():
            z.write(p, rel(p))
print({'step': STEP_NAME, 'tuned_jobs': len(jobs), 'trial_rows': len(trial_summary), 'oof_rows': len(oof_predictions), 'final_check_failures': int((final_checks['status'] == 'FAIL').sum()), 'zip_path': str(ZIP_PATH)})


Exception in thread Thread-4729 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
UnicodeDecodeError: 'cp949' codec can't decode byte 0xec in position 578: illegal multibyte sequence


  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1538, in _execute_c

{'step': '14x_lightweight_candidate_tuning_260516', 'tuned_jobs': 16, 'trial_rows': 480, 'oof_rows': 276948, 'final_check_failures': 0, 'zip_path': 'C:\\Code\\ott-churn-prediction\\park.ingyeom\\zip\\14x_lightweight_candidate_tuning_260516_review_package.zip'}
